# Agent: Humans Producer (Phase 5, MQTT Publish)

This notebook now includes Phases 1–5 producer scope:
- local movement in a 1000x1000 meter area
- boundary reflection with overshoot mirroring
- illegal event generation (`event_type="illegal_act"`)
- max 1 event per human per step
- 1 Hz simulation cadence
- MQTT publishing to `simcity/surveillance/humans/state`
- MQTT publishing to `simcity/surveillance/events/illegal`

Connection policy: if MQTT is unavailable, local simulation continues without publishing.

# Agent: Humans Producer (Phase 5, MQTT Publish)

This notebook now includes Phases 1–5 producer scope:
- local movement in a 1000x1000 meter area
- boundary reflection with overshoot mirroring
- illegal event generation (`event_type=\"illegal_act\"`)
- max 1 event per human per step
- 1 Hz simulation cadence
- MQTT publishing to `simcity/surveillance/humans/state`
- MQTT publishing to `simcity/surveillance/events/illegal`

Connection policy: if MQTT is unavailable, local simulation continues without publishing.

In [13]:
# Setup / imports
from __future__ import annotations

# Standard-library imports used for simulation math/timing/randomization
import colorsys
import json
import math
import random
import time
from dataclasses import dataclass
from typing import Any

# Notebook helper to render widgets/maps inside the output area
from IPython.display import display

# anymap-ts map widget for live map visualization
from anymap_ts import Map
# Project config loader (reads config.yaml and env vars)
from simulated_city.config import load_config
# MQTT helpers used in Phase 5 publishing
from simulated_city.mqtt import MqttConnector, MqttPublisher

In [14]:
# Config load + runtime settings

# Load shared project configuration from config.yaml/.env
cfg = load_config()
sim_cfg = cfg.simulation

# Parken Stadion (Copenhagen) center used as visual anchor for the map
PARKEN_LAT = 55.7029
PARKEN_LON = 12.5720

# Local simulation area boundaries in meters (0..1000 on both axes)
AREA_MIN = 0.0
AREA_MAX = 1000.0

# Design-locked constants for this project
STEP_LENGTH_M = 7.0
HUMAN_COUNT = 10
TOTAL_STEPS = 60
EVENT_TYPE = "illegal_act"

# Runtime settings now come from config where available (Phase 2)
EVENT_PROBABILITY = float(sim_cfg.arrival_prob) if sim_cfg is not None else 0.25

# Keep wall-clock pacing at 1s by default, but allow config override if > 0
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0

# Topic policy (Phase 4/5): surveillance root is design-locked
SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_HUMANS_STATE = f"{SURVEILLANCE_TOPIC_ROOT}/humans/state"
TOPIC_EVENTS_ILLEGAL = f"{SURVEILLANCE_TOPIC_ROOT}/events/illegal"

# Locked live-stream settings from project protocol
LIVE_QOS = 0
LIVE_RETAIN = False

# Keep simulation non-reproducible: do NOT set random.seed(...)
print(f"Config loaded. Primary MQTT host: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Runtime settings -> event_prob={EVENT_PROBABILITY}, steps={TOTAL_STEPS}, delay={STEP_DELAY_S}s")
print(
    f"Topic policy -> design_root={SURVEILLANCE_TOPIC_ROOT}, "
    f"config_base_topic={cfg.mqtt.base_topic} (informational only)"
 )

Config loaded. Primary MQTT host: 127.0.0.1:1883
Runtime settings -> event_prob=0.25, steps=60, delay=1.0s
Topic policy -> design_root=simcity/surveillance, config_base_topic=simulated-city (informational only)


In [15]:
# Helper functions: local meter coordinates, boundary reflection, and colors

def local_xy_to_lnglat(x_m: float, y_m: float, center_lat: float, center_lon: float) -> tuple[float, float]:
    """Approximate local meters (x east, y north) to (lng, lat)."""
    # Approximate meters-per-degree values; sufficient for small local offsets.
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lon = 111_320.0 * math.cos(math.radians(center_lat))

    # Convert local meter offsets to geographic deltas.
    lat = center_lat + (y_m / meters_per_deg_lat)
    lon = center_lon + (x_m / meters_per_deg_lon)
    return lon, lat


def reflect_axis(position: float, delta: float, low: float = AREA_MIN, high: float = AREA_MAX) -> tuple[float, float]:
    """Reflect one axis and mirror overshoot into [low, high]."""
    # First apply the proposed movement on this axis.
    new_position = position + delta
    new_delta = delta

    # If we left bounds, mirror overshoot back in and flip direction.
    # Loop handles rare large overshoot values robustly.
    while new_position < low or new_position > high:
        if new_position > high:
            overshoot = new_position - high
            new_position = high - overshoot
            new_delta = -new_delta
        elif new_position < low:
            overshoot = low - new_position
            new_position = low + overshoot
            new_delta = -new_delta

    return new_position, new_delta


def generate_unique_colors(count: int) -> list[str]:
    """Create random-but-unique colors for human markers."""
    # Start with evenly spaced hues so colors are distinct.
    hues = [index / count for index in range(count)]
    # Shuffle to avoid predictable color ordering between humans.
    random.shuffle(hues)

    colors: list[str] = []
    for hue in hues:
        # Convert HSV to RGB for vivid, readable marker colors.
        r, g, b = colorsys.hsv_to_rgb(hue, 0.75, 0.95)
        colors.append(f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")

    return colors

In [16]:
# Human model + initialization

# Dataclass keeps each simulated human's state in one simple object.
@dataclass
class Human:
    human_id: str
    name: str
    x: float
    y: float
    marker_color: str
    criminal_count: int = 0


# Fixed list of 10 names (design requires unique full-name IDs).
NAMES = [
    "Ava Jensen",
    "Liam Nielsen",
    "Emma Larsen",
    "Noah Pedersen",
    "Sofia Madsen",
    "Oliver Christensen",
    "Clara Andersen",
    "William Thomsen",
    "Freja Rasmussen",
    "Lucas Mortensen",
]

# Build one unique marker color per human.
colors = generate_unique_colors(HUMAN_COUNT)
humans: list[Human] = []
for index in range(HUMAN_COUNT):
    humans.append(
        Human(
            human_id=NAMES[index],
            name=NAMES[index],
            # Start each human at a random position inside [0, 1000] x [0, 1000].
            x=random.uniform(AREA_MIN, AREA_MAX),
            y=random.uniform(AREA_MIN, AREA_MAX),
            marker_color=colors[index],
        )
    )

# Print initialized state so you can inspect the random start distribution.
print("Initialized humans:")
for human in humans:
    print(f"- {human.human_id}: x={human.x:.1f}, y={human.y:.1f}, color={human.marker_color}")

Initialized humans:
- Ava Jensen: x=725.4, y=394.4, color=#60f23c
- Liam Nielsen: x=647.1, y=877.0, color=#603cf2
- Emma Larsen: x=846.6, y=703.6, color=#f2a93c
- Noah Pedersen: x=523.6, y=490.2, color=#cdf23c
- Sofia Madsen: x=44.4, y=877.8, color=#f23c3c
- Oliver Christensen: x=694.5, y=761.8, color=#3c85f2
- Clara Andersen: x=71.6, y=227.1, color=#cd3cf2
- William Thomsen: x=180.4, y=654.3, color=#f23ca9
- Freja Rasmussen: x=856.9, y=951.3, color=#3cf2f2
- Lucas Mortensen: x=845.1, y=492.4, color=#3cf285


In [17]:
# Map setup (anymap)

# Create map centered on Parken for visual context.
m = Map(center=(PARKEN_LON, PARKEN_LAT), zoom=15.8, height="650px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")

# Render the initial human positions as map markers.
for human in humans:
    # Convert local simulation coordinates to approximate lng/lat around map center.
    lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
    # Use human_id as stable marker name so updates can target the same person.
    m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

# Show map widget in notebook output.
display(m)
print("Map displayed. Run the simulation cell to start.")

Map displayed. Run the simulation cell to start.


In [19]:
# 1 Hz simulation loop: movement + event generation + MQTT publishing

# Dictionary keyed by step -> list of events generated in that step.
events_by_step: dict[int, list[dict[str, Any]]] = {}

# Phase 5 MQTT initialization: continue locally if broker is unavailable.
mqtt_connector: MqttConnector | None = None
mqtt_publisher: MqttPublisher | None = None
publishing_enabled = False

try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="humans-producer")
    mqtt_connector.connect()
    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_publisher = MqttPublisher(mqtt_connector)
        publishing_enabled = True
        print(f"MQTT publishing enabled -> {TOPIC_HUMANS_STATE}, {TOPIC_EVENTS_ILLEGAL}")
    else:
        print("MQTT connection timeout. Continuing local simulation without publishing.")
except Exception as exc:
    print(f"MQTT unavailable ({exc}). Continuing local simulation without publishing.")

for step in range(TOTAL_STEPS):
    # Timestamp start of loop iteration to maintain near-1Hz pacing.
    step_started = time.perf_counter()
    step_events: list[dict[str, Any]] = []

    for human in humans:
        # Continuous movement with random direction each step.
        angle = random.uniform(0.0, 2.0 * math.pi)
        dx = STEP_LENGTH_M * math.cos(angle)
        dy = STEP_LENGTH_M * math.sin(angle)

        # Reflect independently per axis with mirrored overshoot.
        human.x, _ = reflect_axis(human.x, dx)
        human.y, _ = reflect_axis(human.y, dy)

        # Update marker position to match new simulated coordinates.
        lng, lat = local_xy_to_lnglat(human.x - 500.0, human.y - 500.0, PARKEN_LAT, PARKEN_LON)
        m.remove_marker(human.human_id)
        m.add_marker(lng, lat, name=human.human_id, color=human.marker_color, popup=human.name)

        # Publish live movement state (minimum required fields).
        if publishing_enabled and mqtt_publisher is not None:
            state_payload = {
                "step": step,
                "human_id": human.human_id,
                "name": human.name,
                "x": human.x,
                "y": human.y,
            }
            try:
                mqtt_publisher.publish_json(
                    TOPIC_HUMANS_STATE,
                    json.dumps(state_payload),
                    qos=LIVE_QOS,
                    retain=LIVE_RETAIN,
                )
            except Exception as exc:
                publishing_enabled = False
                print(f"MQTT publish disabled after error: {exc}")

        # Illegal event generation (max 1 event per human per step).
        if random.random() < EVENT_PROBABILITY:
            event_payload = {
                "step": step,
                # Contracted event_id format from design spec.
                "event_id": f"{step}:{human.human_id}",
                "human_id": human.human_id,
                "name": human.name,
                "x": human.x,
                "y": human.y,
                "event_type": EVENT_TYPE,
            }
            step_events.append(event_payload)

            # Publish illegal events only when they are created.
            if publishing_enabled and mqtt_publisher is not None:
                try:
                    mqtt_publisher.publish_json(
                        TOPIC_EVENTS_ILLEGAL,
                        json.dumps(event_payload),
                        qos=LIVE_QOS,
                        retain=LIVE_RETAIN,
                    )
                except Exception as exc:
                    publishing_enabled = False
                    print(f"MQTT publish disabled after error: {exc}")

    # Save all events generated in the current timestep.
    events_by_step[step] = step_events
    print(f"step={step:03d} events={len(step_events)}")

    # Sleep only the remaining time to keep total loop time near 1 second.
    elapsed = time.perf_counter() - step_started
    time.sleep(max(0.0, STEP_DELAY_S - elapsed))

# Final cleanup step: disconnect MQTT if we connected successfully.
if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()
    print("MQTT connector disconnected.")

print("Phase 5 simulation complete.")

MQTT publishing enabled -> simcity/surveillance/humans/state, simcity/surveillance/events/illegal
step=000 events=1
step=001 events=3
step=002 events=4
step=003 events=2
step=004 events=3
step=005 events=3
step=006 events=1
step=007 events=3
step=008 events=1
step=009 events=3
step=010 events=1
step=011 events=2
step=012 events=1
step=013 events=1
step=014 events=1
step=015 events=3
step=016 events=2
step=017 events=3
step=018 events=1
step=019 events=3
step=020 events=4
step=021 events=3
step=022 events=2
step=023 events=2
step=024 events=5
step=025 events=2
step=026 events=4
step=027 events=5
step=028 events=2
step=029 events=4
step=030 events=2
step=031 events=0
step=032 events=2
step=033 events=2
step=034 events=7
step=035 events=3
step=036 events=2
step=037 events=0
step=038 events=4
step=039 events=4
step=040 events=2
step=041 events=2
step=042 events=2
step=043 events=2
step=044 events=0
step=045 events=1
step=046 events=3
step=047 events=0
step=048 events=1
step=049 events=2
st

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


MQTT connector disconnected.
Phase 5 simulation complete.


In [45]:
# Quick verification checks for Phase 5

# Flatten event list across all steps to evaluate uniqueness contract.
all_event_ids = [event["event_id"] for events in events_by_step.values() for event in events]
unique_event_ids = set(all_event_ids)

# Compare counts: if equal, there are no duplicate event IDs.
print(f"Total events: {len(all_event_ids)}")
print(f"Unique event_ids: {len(unique_event_ids)}")
print(f"event_id uniqueness ok: {len(all_event_ids) == len(unique_event_ids)}")

# Print a small sample for manual inspection.
if all_event_ids:
    print("Sample event_ids:")
    for event_id in list(unique_event_ids)[:10]:
        print(" -", event_id)

Total events: 122
Unique event_ids: 122
event_id uniqueness ok: True
Sample event_ids:
 - 12:Ava Jensen
 - 38:William Thomsen
 - 42:William Thomsen
 - 5:William Thomsen
 - 52:Emma Larsen
 - 55:Emma Larsen
 - 46:Clara Andersen
 - 6:Clara Andersen
 - 25:Noah Pedersen
 - 11:Freja Rasmussen
